# Bài 2: Phân lớp chữ số viết tay MNIST

**Họ tên:** Lê Nguyễn Anh Du  
**MSSV:** 6451071011

Xây dựng Neural Network bằng Keras với TensorFlow backend để phân lớp ảnh chữ số viết tay MNIST.

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

HO_TEN = "Lê Nguyễn Anh Du"
MSSV = "6451071011"
def in_thong_tin():
    print(f"Họ tên: {HO_TEN} | MSSV: {MSSV}")

np.random.seed(42)
tf.random.set_seed(42)
in_thong_tin()
print("Đã nhập thư viện và thiết lập seed.")

## a) Đọc dữ liệu train và test MNIST từ Keras

In [ ]:
in_thong_tin()
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print("Đã tải dữ liệu MNIST từ Keras.")

## b) In kích thước tập train và test

In [ ]:
in_thong_tin()
print("Kích thước train_images:", train_images.shape)
print("Kích thước train_labels:", train_labels.shape)
print("Kích thước test_images:", test_images.shape)
print("Kích thước test_labels:", test_labels.shape)

## c) Chuẩn hóa pixel về đoạn [0, 1]

In [ ]:
in_thong_tin()
train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0
print("Giá trị pixel sau chuẩn hóa — train:", train_images.min(), "đến", train_images.max())
print("Giá trị pixel sau chuẩn hóa — test:", test_images.min(), "đến", test_images.max())

## d) Chia tập train thành train và validation theo tỷ lệ 8:2

In [ ]:
in_thong_tin()
train_images, validation_images, train_labels, validation_labels = train_test_split(
    train_images, train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)
print("Kích thước train_images:", train_images.shape)
print("Kích thước train_labels:", train_labels.shape)
print("Kích thước validation_images:", validation_images.shape)
print("Kích thước validation_labels:", validation_labels.shape)

## e) Xây dựng Neural Network

Mô hình gồm Flatten, Dense 128 neuron với ReLU, Dropout 20% và Dense đầu ra 10 lớp với Softmax.

In [ ]:
in_thong_tin()
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(10, activation="softmax")
])
model.summary()

## f) Biên dịch mô hình

In [ ]:
in_thong_tin()
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
print("Đã biên dịch mô hình với Adam, sparse categorical crossentropy và accuracy.")

## g) Huấn luyện mô hình

In [ ]:
in_thong_tin()
history = model.fit(
    train_images, train_labels,
    validation_data=(validation_images, validation_labels),
    epochs=5,
    batch_size=32,
    verbose=1
)

## h) Vẽ accuracy và loss sau mỗi epoch

In [ ]:
in_thong_tin()
epochs = range(1, len(history.history["accuracy"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history.history["accuracy"], marker="o", label="Train")
axes[0].plot(epochs, history.history["val_accuracy"], marker="o", label="Validation")
axes[0].set_title("Accuracy theo epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)
axes[1].plot(epochs, history.history["loss"], marker="o", label="Train")
axes[1].plot(epochs, history.history["val_loss"], marker="o", label="Validation")
axes[1].set_title("Loss theo epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

## i) Dự đoán phân lớp cho tập test

In [ ]:
in_thong_tin()
test_probabilities = model.predict(test_images, batch_size=256, verbose=1)
test_predictions = np.argmax(test_probabilities, axis=1)
print("Số dự đoán:", len(test_predictions))
print("5 nhãn dự đoán đầu tiên:", test_predictions[:5])
print("5 nhãn thực tế đầu tiên:", test_labels[:5])

## j) Tính Accuracy trên tập test

In [ ]:
in_thong_tin()
test_accuracy = accuracy_score(test_labels, test_predictions)
print(f"Accuracy trên tập test: {test_accuracy:.4f}")

## k) Precision, Recall và F1-score

In [ ]:
in_thong_tin()
print(classification_report(
    test_labels,
    test_predictions,
    labels=np.arange(10),
    target_names=[str(i) for i in range(10)],
    digits=4,
    zero_division=0
))

## l) Confusion matrix dạng heatmap

In [ ]:
in_thong_tin()
cm = confusion_matrix(test_labels, test_predictions, labels=np.arange(10))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=np.arange(10), yticklabels=np.arange(10))
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.title("Confusion matrix — MNIST")
plt.tight_layout()
plt.show()

## m) Kết quả phân lớp của 5 ảnh đầu tiên trong tập test

In [ ]:
in_thong_tin()
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for image_index in range(5):
    image = np.expand_dims(test_images[image_index], axis=0)
    probabilities = model.predict(image, verbose=0)
    predicted_label = int(np.argmax(probabilities, axis=1)[0])
    true_label = int(test_labels[image_index])
    print(f"Ảnh {image_index}: dự đoán = {predicted_label}, nhãn thực tế = {true_label}")
    axes[image_index].imshow(test_images[image_index], cmap="gray")
    axes[image_index].set_title(f"Dự đoán: {predicted_label}\nThực tế: {true_label}")
    axes[image_index].axis("off")
plt.tight_layout()
plt.show()

## n) 5 ảnh đầu tiên phân lớp sai trong tập test

In [ ]:
in_thong_tin()
wrong_indices = np.where(test_predictions != test_labels)[0][:5]
if len(wrong_indices) == 0:
    print("Không có ảnh phân lớp sai trong tập test.")
else:
    print("Chỉ số của các ảnh phân lớp sai đầu tiên:", wrong_indices)
    fig, axes = plt.subplots(1, len(wrong_indices), figsize=(3 * len(wrong_indices), 3))
    if len(wrong_indices) == 1:
        axes = [axes]
    for ax, image_index in zip(axes, wrong_indices):
        predicted_label = int(test_predictions[image_index])
        true_label = int(test_labels[image_index])
        ax.imshow(test_images[image_index], cmap="gray")
        ax.set_title(f"Dự đoán: {predicted_label}\nThực tế: {true_label}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()